### Allscripts Sunrise (SCM) Drug Exposure Hydration

Runnable-only version.

Notes:
- Reset is active at the top for clean reruns.
- Drug concepts resolve directly from `dbo_sxammgenericitem.RxNormCode` into OMOP concept tables.
- Route concepts still use `domain_source_to_concept` when available; unmapped routes stay `0`.


In [ ]:
%sql
TRUNCATE TABLE _exponent.omop_scm.drug_exposure;

DELETE FROM _exponent.omop_silver.drug_exposure
WHERE source_system = 'allscripts_scm';

DELETE FROM _exponent.omop_mapping.source_to_drug_exposure
WHERE source_system = 'allscripts_scm';

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_drug_exposure AS
SELECT
  COALESCE(rxnorm_standard.concept_id, rxnorm_source.concept_id, 0) AS drug_concept_id,
  DATE(COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)) AS drug_exposure_start_date,
  COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS drug_exposure_start_datetime,
  CASE
    WHEN ord.StopDtm IS NOT NULL THEN DATE(ord.StopDtm)
    ELSE DATE(COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen))
  END AS drug_exposure_end_date,
  CASE
    WHEN ord.StopDtm IS NOT NULL THEN ord.StopDtm
    ELSE COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)
  END AS drug_exposure_end_datetime,
  CASE WHEN ord.StopDtm IS NOT NULL THEN DATE(ord.StopDtm) ELSE NULL END AS verbatim_end_date,
  32817 AS drug_type_concept_id,
  NULL AS stop_reason,
  medext.NumRefills AS refills,
  TRY_CAST(medext.DispenseAmount AS DOUBLE) AS quantity,
  NULL AS days_supply,
  medext.RxInstructions AS sig,
  COALESCE(route_concept.omop_concept_id, 0) AS route_concept_id,
  NULL AS lot_number,
  COALESCE(medext.OrderedAsDisplay, gi.GenericItemName, ord.Name) AS drug_source_value,
  COALESCE(rxnorm_source.concept_id, 0) AS drug_source_concept_id,
  medext.OrderRouteCode AS route_source_value,
  COALESCE(medext.DosageLow, medext.Uom) AS dose_unit_source_value,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING)) AS person_source_value,
  CASE
    WHEN ord.CareProviderGUID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3careprovider', 'GUID', CAST(ord.CareProviderGUID AS STRING))
    ELSE NULL
  END AS provider_source_value,
  CASE
    WHEN ord.ClientVisitGUID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3clientvisit', 'GUID', CAST(ord.ClientVisitGUID AS STRING))
    ELSE NULL
  END AS visit_occurrence_source_value,
  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3order', 'GUID', CAST(ord.GUID AS STRING)) AS drug_exposure_source_value,
  'allscripts_scm' AS source_system
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON gi.GenericItemID = medext.PrescriptionGenericItemID
 AND gi.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent.omop.concept rxnorm_source
  ON rxnorm_source.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
 AND rxnorm_source.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
 AND rxnorm_source.domain_id = 'Drug'
 AND rxnorm_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship rxnorm_maps_to
  ON rxnorm_maps_to.concept_id_1 = rxnorm_source.concept_id
 AND rxnorm_maps_to.relationship_id = 'Maps to'
 AND rxnorm_maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept rxnorm_standard
  ON rxnorm_standard.concept_id = rxnorm_maps_to.concept_id_2
 AND rxnorm_standard.standard_concept = 'S'
 AND rxnorm_standard.domain_id = 'Drug'
 AND rxnorm_standard.invalid_reason IS NULL
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept route_concept
  ON route_concept.source_id = medext.OrderRouteCode
 AND route_concept.domain_id = 'Route'
 AND route_concept.source_system = 'allscripts_scm'
 AND route_concept.active_flag = 1
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.drug_exposure AS t
USING (
  SELECT * FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY drug_exposure_source_value, drug_type_concept_id
        ORDER BY drug_exposure_start_date DESC
      ) AS rn
    FROM silver_drug_exposure
  ) WHERE rn = 1
) AS s
ON t.drug_exposure_source_value = s.drug_exposure_source_value
 AND t.drug_type_concept_id = s.drug_type_concept_id

WHEN MATCHED AND (
     NOT (t.drug_concept_id <=> s.drug_concept_id)
  OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
  OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
  OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
  OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
  OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
  OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.refills <=> s.refills)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.days_supply <=> s.days_supply)
  OR NOT (t.sig <=> s.sig)
  OR NOT (t.route_concept_id <=> s.route_concept_id)
  OR NOT (t.lot_number <=> s.lot_number)
  OR NOT (t.drug_source_value <=> s.drug_source_value)
  OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
  OR NOT (t.route_source_value <=> s.route_source_value)
  OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.visit_detail_source_value <=> s.visit_detail_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.drug_concept_id = s.drug_concept_id,
  t.drug_exposure_start_date = s.drug_exposure_start_date,
  t.drug_exposure_start_datetime = s.drug_exposure_start_datetime,
  t.drug_exposure_end_date = s.drug_exposure_end_date,
  t.drug_exposure_end_datetime = s.drug_exposure_end_datetime,
  t.verbatim_end_date = s.verbatim_end_date,
  t.drug_type_concept_id = s.drug_type_concept_id,
  t.stop_reason = s.stop_reason,
  t.refills = s.refills,
  t.quantity = s.quantity,
  t.days_supply = s.days_supply,
  t.sig = s.sig,
  t.route_concept_id = s.route_concept_id,
  t.lot_number = s.lot_number,
  t.drug_source_value = s.drug_source_value,
  t.drug_source_concept_id = s.drug_source_concept_id,
  t.route_source_value = s.route_source_value,
  t.dose_unit_source_value = s.dose_unit_source_value,
  t.person_source_value = s.person_source_value,
  t.provider_source_value = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value = s.visit_detail_source_value,
  t.source_system = s.source_system,
  t.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  drug_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.drug_concept_id,
  s.drug_exposure_start_date,
  s.drug_exposure_start_datetime,
  s.drug_exposure_end_date,
  s.drug_exposure_end_datetime,
  s.verbatim_end_date,
  s.drug_type_concept_id,
  s.stop_reason,
  s.refills,
  s.quantity,
  s.days_supply,
  s.sig,
  s.route_concept_id,
  s.lot_number,
  s.drug_source_value,
  s.drug_source_concept_id,
  s.route_source_value,
  s.dose_unit_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.drug_exposure_source_value,
  s.source_system,
  CURRENT_TIMESTAMP()
);

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
  source_system,
  drug_exposure_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp
)
SELECT
  s.source_system,
  s.drug_exposure_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP())
FROM (
  SELECT DISTINCT source_system, drug_exposure_source_value, last_mod_tsp
  FROM _exponent.omop_silver.drug_exposure
  WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
  ON s.drug_exposure_source_value = x.drug_exposure_source_value;

In [ ]:
%sql
MERGE INTO _exponent.omop_scm.drug_exposure AS gold
USING (
  SELECT
    sde.drug_exposure_id,
    stp.person_id,
    s.drug_concept_id,
    s.drug_exposure_start_date,
    s.drug_exposure_start_datetime,
    s.drug_exposure_end_date,
    s.drug_exposure_end_datetime,
    s.verbatim_end_date,
    s.drug_type_concept_id,
    s.stop_reason,
    s.refills,
    s.quantity,
    s.days_supply,
    s.sig,
    s.route_concept_id,
    s.lot_number,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.drug_source_value,
    s.drug_source_concept_id,
    s.route_source_value,
    s.dose_unit_source_value
  FROM _exponent.omop_silver.drug_exposure s
  JOIN _exponent.omop_mapping.source_to_drug_exposure sde
    ON sde.drug_exposure_source_value = s.drug_exposure_source_value
   AND sde.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.drug_concept_id = src.drug_concept_id,
  gold.drug_exposure_start_date = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime = src.drug_exposure_end_datetime,
  gold.verbatim_end_date = src.verbatim_end_date,
  gold.drug_type_concept_id = src.drug_type_concept_id,
  gold.stop_reason = src.stop_reason,
  gold.refills = src.refills,
  gold.quantity = src.quantity,
  gold.days_supply = src.days_supply,
  gold.sig = src.sig,
  gold.route_concept_id = src.route_concept_id,
  gold.lot_number = src.lot_number,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.drug_source_value = src.drug_source_value,
  gold.drug_source_concept_id = src.drug_source_concept_id,
  gold.route_source_value = src.route_source_value,
  gold.dose_unit_source_value = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);